# 01 Data Forensics

**Purpose:** Inspect raw competition datasets, run the Bronze -> Silver -> Rejected pipeline, and document the data quality issues that must be neutralized before POI enrichment and modeling.

This notebook is report-facing: reusable logic lives in `src/data/` and `src/quality/`, while this notebook shows the evidence, counts, and assertions.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.silver_pipeline import PipelinePaths, run_pipeline

RAW = PROJECT_ROOT / "data" / "raw"
BRONZE = PROJECT_ROOT / "data" / "bronze"
SILVER = PROJECT_ROOT / "data" / "silver"
REJECTED = PROJECT_ROOT / "data" / "rejected"
REPORTS_EDA = PROJECT_ROOT / "reports" / "eda"

PIPELINE_PATHS = PipelinePaths(
    raw=RAW,
    bronze=BRONZE,
    silver=SILVER,
    rejected=REJECTED,
    reports_eda=REPORTS_EDA,
)

## 1. Raw Data Inventory

In [2]:
raw_inventory = []
for path in sorted(RAW.glob("*.csv")):
    df = pd.read_csv(path)
    raw_inventory.append({
        "dataset": path.name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
        "columns_list": ", ".join(df.columns),
    })

raw_inventory_df = pd.DataFrame(raw_inventory)
raw_inventory_df

,dataset,rows,columns,duplicate_rows,columns_list
0,distributor_seasonality_details.csv,360,4,0,"Distributor_ID, Year, Month, Seasonality_Index"
1,holiday_list.csv,349,3,93,"Date, Holiday_Name, Holiday_Type"
2,outlet_coordinates.csv,20000,3,0,"Outlet_ID, Latitude, Longitude"
3,outlet_master.csv,20000,4,0,"Outlet_ID, Outlet_Size, Cooler_Count, Outlet_Type"
4,transactions_history_final.csv,2376389,7,0,"Outlet_ID, Year, Month, Distributor_ID, SKU_ID..."


## 2. First-Pass Forensic Signals

In [3]:
outlet_master_raw = pd.read_csv(RAW / "outlet_master.csv")
coords_raw = pd.read_csv(RAW / "outlet_coordinates.csv")
transactions_raw = pd.read_csv(RAW / "transactions_history_final.csv")
holidays_raw = pd.read_csv(RAW / "holiday_list.csv")

forensics = {
    "outlet_size_missing": int(outlet_master_raw["Outlet_Size"].isna().sum()),
    "outlet_size_values": outlet_master_raw["Outlet_Size"].value_counts(dropna=False).to_dict(),
    "outlet_type_values": outlet_master_raw["Outlet_Type"].value_counts(dropna=False).to_dict(),
    "invalid_coordinate_bbox_rows": int((
        (coords_raw["Latitude"] < 5) | (coords_raw["Latitude"] > 10) |
        (coords_raw["Longitude"] < 79) | (coords_raw["Longitude"] > 82)
    ).sum()),
    "nonpositive_volume_rows": int((transactions_raw["Volume_Liters"] <= 0).sum()),
    "nonpositive_bill_rows": int((transactions_raw["Total_Bill_Value"] <= 0).sum()),
    "transaction_composite_key_duplicates": int(transactions_raw.duplicated([
        "Outlet_ID", "Year", "Month", "Distributor_ID", "SKU_ID"
    ]).sum()),
    "holiday_exact_duplicates": int(holidays_raw.duplicated(["Date", "Holiday_Name", "Holiday_Type"]).sum()),
}

forensics

{'outlet_size_missing': 196,
 'outlet_size_values': {'Small': 9672,
  'Medium': 5702,
  'Large': 2887,
  'Extra Large': 943,
  'small': 600,
  nan: 196},
 'outlet_type_values': {'Hotel': 2797,
  'Grocery': 2768,
  'SMMT': 2723,
  'Pharmacy': 2691,
  'Kiosk': 2691,
  'Bakery': 2678,
  'Eatery': 2667,
  'Bakry': 395,
  'Grocry': 390,
  ' Eatery ': 200},
 'invalid_coordinate_bbox_rows': 240,
 'nonpositive_volume_rows': 4853,
 'nonpositive_bill_rows': 4753,
 'transaction_composite_key_duplicates': 32240,
 'holiday_exact_duplicates': 93}

## 3. Run Bronze -> Silver -> Rejected Pipeline

In [4]:
metrics = run_pipeline(PIPELINE_PATHS)
pd.DataFrame(metrics).T

,raw_rows,bronze_rows,silver_rows,hard_rejected_rows,hard_rejected_events,warning_rows,warning_events,corrected_rows,action_taken
outlet_master,20000,20000,20000,0,0,196,196,0,Standardized outlet categories; retained missi...
outlet_coordinates,20000,20000,19960,40,80,0,0,200,Corrected safe lat/lon swaps; removed unusable...
distributor_seasonality_details,360,360,360,0,0,0,0,0,Canonicalized seasonality labels and validated...
holiday_list,349,349,256,93,93,0,0,0,Removed exact duplicate holidays and preserved...
transactions_history_final,2376389,2376389,2339409,36980,41846,0,0,0,"Rejected non-positive values, duplicate compos..."


## 4. Rejected Record Store

In [5]:
rejection_summary = []
for path in sorted(REJECTED.glob("*_rejected.csv")):
    df = pd.read_csv(path)
    if df.empty:
        rejection_summary.append({"file": path.name, "check_name": "<none>", "rejected_events": 0})
    else:
        counts = df["check_name"].value_counts().reset_index()
        counts.columns = ["check_name", "rejected_events"]
        counts.insert(0, "file", path.name)
        rejection_summary.extend(counts.to_dict("records"))

rejection_summary_df = pd.DataFrame(rejection_summary)

warning_summary = []
for path in sorted(REJECTED.glob("*_warnings.csv")):
    df = pd.read_csv(path)
    if df.empty:
        warning_summary.append({"file": path.name, "check_name": "<none>", "warning_events": 0})
    else:
        counts = df["check_name"].value_counts().reset_index()
        counts.columns = ["check_name", "warning_events"]
        counts.insert(0, "file", path.name)
        warning_summary.extend(counts.to_dict("records"))

warning_summary_df = pd.DataFrame(warning_summary)
rejection_summary_df, warning_summary_df

(                                           file  \
 0  distributor_seasonality_details_rejected.csv   
 1                     holiday_list_rejected.csv   
 2               outlet_coordinates_rejected.csv   
 3               outlet_coordinates_rejected.csv   
 4                    outlet_master_rejected.csv   
 5       transactions_history_final_rejected.csv   
 6       transactions_history_final_rejected.csv   
 7       transactions_history_final_rejected.csv   
 
                             check_name  rejected_events  
 0                               <none>                0  
 1              holiday_exact_duplicate               93  
 2              latitude_sri_lanka_bbox               40  
 3             longitude_sri_lanka_bbox               40  
 4                               <none>                0  
 5  transaction_composite_key_duplicate            32240  
 6                      volume_positive             4853  
 7                  bill_value_positive             4753  

In [6]:
corrections = pd.read_csv(SILVER / "outlet_coordinates_corrections.csv")
corrections.head(), len(corrections)

(   Outlet_ID  Original_Latitude  Original_Longitude  Corrected_Latitude  \
 0  OUT_00030          79.923311            7.188307            7.188307   
 1  OUT_00171          79.999061            6.706547            6.706547   
 2  OUT_00252          80.032979            7.134312            7.134312   
 3  OUT_00457          80.008335            6.712571            6.712571   
 4  OUT_00511          80.019915            6.866704            6.866704   
 
    Corrected_Longitude                                  correction_reason  \
 0            79.923311  Latitude/longitude appeared swapped and correc...   
 1            79.999061  Latitude/longitude appeared swapped and correc...   
 2            80.032979  Latitude/longitude appeared swapped and correc...   
 3            80.008335  Latitude/longitude appeared swapped and correc...   
 4            80.019915  Latitude/longitude appeared swapped and correc...   
 
                 corrected_at  
 0  2026-05-15T18:12:17+00:00  
 1  2026

## 5. Silver Layer Acceptance Checks

In [7]:
outlet_master = pd.read_csv(SILVER / "outlet_master.csv")
coords = pd.read_csv(SILVER / "outlet_coordinates.csv")
transactions = pd.read_csv(SILVER / "transactions_history_final.csv")
seasonality = pd.read_csv(SILVER / "distributor_seasonality_details.csv")
holidays = pd.read_csv(SILVER / "holiday_list.csv")

expected_distributors = {
    "DIST_W_01", "DIST_W_02", "DIST_W_03",
    "DIST_C_01", "DIST_C_02", "DIST_C_03",
    "DIST_NW_01", "DIST_NW_02",
    "DIST_S_01", "DIST_S_02",
}
valid_coord_statuses = {"valid", "corrected", "quarantined", "missing"}
valid_outlet_size_statuses = {"provided", "missing"}
canonical_seasonality = {"Moderate", "Favorable", "Un-Favorable"}

assert outlet_master["Outlet_ID"].is_unique
assert len(outlet_master) == outlet_master_raw["Outlet_ID"].nunique()
assert set(outlet_master["outlet_size_status"]).issubset(valid_outlet_size_statuses)
assert (outlet_master.loc[outlet_master["outlet_size_status"].eq("missing"), "Outlet_Size"] == "Unknown").all()
assert not outlet_master.loc[outlet_master["outlet_size_status"].eq("provided"), "Outlet_Size"].eq("Unknown").any()
assert outlet_master["coord_status"].notna().all()
assert set(outlet_master["coord_status"]).issubset(valid_coord_statuses)
assert set(coords["coord_status"]).issubset({"valid", "corrected"})
assert set(outlet_master.loc[outlet_master["has_valid_coord"], "Outlet_ID"]) == set(coords["Outlet_ID"])
assert set(outlet_master.loc[outlet_master["coord_status"].isin({"quarantined", "missing"}), "Outlet_ID"]).isdisjoint(set(coords["Outlet_ID"]))
assert coords["Latitude"].between(5, 10).all()
assert coords["Longitude"].between(79, 82).all()
assert (transactions["Volume_Liters"] > 0).all()
assert (transactions["Total_Bill_Value"] > 0).all()
assert set(transactions["Outlet_ID"]).issubset(set(outlet_master["Outlet_ID"]))
assert set(transactions["Distributor_ID"]).issubset(expected_distributors)
assert set(seasonality["Seasonality_Index"]).issubset(canonical_seasonality)
assert not holidays.duplicated(["Date", "Holiday_Name", "Holiday_Type"]).any()
assert not seasonality.duplicated(["Distributor_ID", "Year", "Month"]).any()

"All Silver acceptance checks passed."

'All Silver acceptance checks passed.'

## 6. Report Notes

- Bronze preserves raw files exactly as provided.
- Silver standardizes outlet categories, parses dates, corrects safe coordinate swaps, and removes records that violate mandatory DQ rules.
- Rejected files preserve every failure event with a reason, so the team can defend all removals and revisit quarantined records if needed.
- The report-ready draft is written to `reports/eda/data_forensics_pipeline.md` when the pipeline runs.